# HALO Harness Optimization

This notebook keeps the existing Databricks agent demo intact, then adds a harness optimization pass with [HALO](https://github.com/context-labs/HALO).

The goal is broader than prompt optimization or skill generation. HALO reviews the same evidence used by skills generation and produces a Codex-ready handoff for improving the whole harness: prompts, tool routing, sufficiency checks, Genie fallback behavior, skills, eval coverage, and deployment guardrails.

Prerequisites:
- Run `00_setup.ipynb` through `09-Evaluation.ipynb` for the existing demo path.
- Run `05-JudgeAlignment.ipynb` so the aligned judge has semantic and episodic memory.
- Run `06-PromptOptimization.ipynb` and `07-AgentSkillsGeneration.ipynb` so optimized prompt and skills artifacts exist.
- Use `gpt-5-4-external` as the model endpoint for HALO. If Databricks OpenAI-compatible routing does not support HALO's Agents SDK calls, this notebook can fall back to direct OpenAI with `OPENAI_API_KEY`.

In [ ]:
import json
from datetime import datetime, timezone

BOOTSTRAP_VOLUME_DIR = "/Volumes/main/at_bat_assistant/agent_skills_gepa/halo"
BOOTSTRAP_STATUS_PATH = f"{BOOTSTRAP_VOLUME_DIR}/run_status.json"
BOOTSTRAP_LOG_PATH = f"{BOOTSTRAP_VOLUME_DIR}/run_log.txt"
payload = {
    "ts": datetime.now(timezone.utc).isoformat(),
    "stage": "bootstrap",
    "message": "Starting HALO dependency install cell",
    "data": {"note": "If this remains the latest status, the notebook is still in %pip install or Python restart."},
}
dbutils.fs.mkdirs(BOOTSTRAP_VOLUME_DIR)
dbutils.fs.put(BOOTSTRAP_STATUS_PATH, json.dumps(payload, indent=2), True)
dbutils.fs.put(BOOTSTRAP_LOG_PATH, json.dumps(payload) + "\n", True)
print(payload)


In [ ]:
%pip install -q --prefer-binary "halo-engine==0.1.10" "mlflow>=3.11.1" "typing_extensions>=4.15.0" "databricks-agents>=1.6.0" openai


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import sys
sys.modules.pop("typing_extensions", None)

import mlflow
from databricks.sdk import WorkspaceClient
from IPython.display import Markdown, display

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]
SKILLS_VOLUME_PATH = CONFIG["skills"]["gepa_volume_path"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
HALO_MODEL = os.getenv("HALO_MODEL", CONFIG["llm"].get("endpoint_name", "gpt-5-4-external")).replace("databricks:/", "")
HALO_DIRECT_OPENAI_MODEL = os.getenv("HALO_DIRECT_OPENAI_MODEL", "gpt-5.4-nano")

os.environ.setdefault("MLFLOW_TRACKING_URI", "databricks")
os.environ.setdefault("MLFLOW_REGISTRY_URI", f"databricks-uc://{CATALOG}")
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri(f"databricks-uc://{CATALOG}")
try:
    mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
except Exception as exc:
    print(f"Skipping active experiment setup; trace search will use explicit experiment location. Reason: {exc}")
w = WorkspaceClient()
ARTIFACT_ROOT = Path("/tmp/atbat_halo")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
HALO_TRACE_PATH = ARTIFACT_ROOT / "halo_traces.jsonl"
HALO_OUTPUT_JSON = ARTIFACT_ROOT / "halo_output.json"
HALO_HANDOFF_MD = ARTIFACT_ROOT / "codex_handoff.md"
HALO_LOG_PATH = ARTIFACT_ROOT / "run_log.txt"
HALO_STATUS_JSON = ARTIFACT_ROOT / "run_status.json"
HALO_VOLUME_DIR = f"{SKILLS_VOLUME_PATH.rstrip('/')}/halo"
HALO_VOLUME_LOG_PATH = f"{HALO_VOLUME_DIR}/run_log.txt"
HALO_VOLUME_STATUS_PATH = f"{HALO_VOLUME_DIR}/run_status.json"
HALO_LOG_PATH.write_text("", encoding="utf-8")

def log_event(stage: str, message: str, **data: Any) -> None:
    payload = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "message": message,
        "data": data,
    }
    line = json.dumps(payload, default=str, ensure_ascii=False)
    print(f"[{payload['ts']}] {stage}: {message} {data if data else ''}")
    with HALO_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
    HALO_STATUS_JSON.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    try:
        dbutils.fs.mkdirs(HALO_VOLUME_DIR)
        dbutils.fs.put(HALO_VOLUME_LOG_PATH, HALO_LOG_PATH.read_text(encoding="utf-8"), True)
        dbutils.fs.put(HALO_VOLUME_STATUS_PATH, HALO_STATUS_JSON.read_text(encoding="utf-8"), True)
    except Exception as exc:
        print(f"Could not persist live HALO log: {type(exc).__name__}: {exc}")

print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"HALO Databricks model endpoint: {HALO_MODEL}")
print(f"HALO direct OpenAI fallback model: {HALO_DIRECT_OPENAI_MODEL}")
print(f"Artifact root: {ARTIFACT_ROOT}")
log_event("init", "HALO notebook initialized", experiment_id=EXPERIMENT_ID, halo_model=HALO_MODEL, artifact_root=str(ARTIFACT_ROOT), volume_dir=HALO_VOLUME_DIR)

## Load The Same Evidence Used For Skills Generation

HALO should optimize the harness from the same evidence set used by `07-AgentSkillsGeneration.ipynb`: UC function signatures, optimized prompt, aligned judge memory, evaluated traces, and generated skills.

In [ ]:
def _safe_json(value: Any, max_chars: int = 20000) -> str:
    try:
        text = json.dumps(value, default=str, ensure_ascii=False)
    except Exception:
        text = str(value)
    if len(text) > max_chars:
        return text[:max_chars] + f"... [truncated {len(text) - max_chars} chars]"
    return text


def _read_volume_text(path: str) -> str:
    try:
        return Path(path).read_text()
    except Exception:
        try:
            return dbutils.fs.head(path, 200000)
        except Exception:
            return ""


def _list_volume_files(path: str, *, recursive: bool = False, exclude_prefixes: tuple[str, ...] = ()) -> list[str]:
    def _is_excluded(candidate: str) -> bool:
        return any(candidate.rstrip('/').startswith(prefix.rstrip('/')) for prefix in exclude_prefixes)

    try:
        entries = dbutils.fs.ls(path)
    except Exception:
        return []

    files = []
    for entry in entries:
        entry_path = entry.path
        if _is_excluded(entry_path):
            continue
        if entry_path.endswith('/'):
            if recursive:
                files.extend(_list_volume_files(entry_path, recursive=True, exclude_prefixes=exclude_prefixes))
        else:
            files.append(entry_path)
    return files


def _extract_user_query(trace) -> str:
    try:
        request = trace.data.request
        if isinstance(request, str):
            request = json.loads(request)
        inputs = request.get("input", request.get("inputs", [])) if isinstance(request, dict) else []
        if isinstance(inputs, dict):
            inputs = inputs.get("input", [])
        if isinstance(inputs, list):
            for msg in inputs:
                if isinstance(msg, dict) and msg.get("role") == "user":
                    return str(msg.get("content", ""))
    except Exception:
        pass
    return ""


def _extract_agent_response(trace) -> str:
    try:
        response = trace.data.response
        if isinstance(response, str):
            response = json.loads(response)
        output = response.get("output", []) if isinstance(response, dict) else []
        texts = []
        for item in output if isinstance(output, list) else []:
            if not isinstance(item, dict):
                continue
            for content in item.get("content", []) or []:
                if isinstance(content, dict) and content.get("type") == "output_text":
                    texts.append(str(content.get("text", "")))
        return "\n".join(t for t in texts if t)
    except Exception:
        return ""


def _extract_tool_calls(trace) -> list[dict[str, Any]]:
    try:
        response = trace.data.response
        if isinstance(response, str):
            response = json.loads(response)
        output = response.get("output", []) if isinstance(response, dict) else []
    except Exception:
        output = []
    calls_by_id = {}
    ordered = []
    for item in output if isinstance(output, list) else []:
        if not isinstance(item, dict):
            continue
        if item.get("type") == "function_call":
            call_id = item.get("call_id") or item.get("id") or str(len(ordered))
            entry = {
                "call_id": call_id,
                "name": item.get("name"),
                "arguments": item.get("arguments"),
                "output": None,
            }
            calls_by_id[call_id] = entry
            ordered.append(entry)
        elif item.get("type") == "function_call_output":
            call_id = item.get("call_id")
            if call_id in calls_by_id:
                calls_by_id[call_id]["output"] = item.get("output")
    return ordered


def _assessment_summary(trace) -> list[dict[str, Any]]:
    rows = []
    for assessment in getattr(trace.info, "assessments", []) or []:
        feedback = getattr(assessment, "feedback", None)
        rows.append({
            "name": getattr(assessment, "name", None),
            "source": getattr(getattr(assessment, "source", None), "source_type", None),
            "value": getattr(feedback, "value", None) if feedback else None,
            "rationale": getattr(assessment, "rationale", None),
        })
    return rows

In [ ]:
log_event("evidence", "Loading UC function descriptions", function_count=len(UC_TOOL_NAMES))
# UC function signatures
uc_functions = []
for fn in UC_TOOL_NAMES:
    try:
        rows = spark.sql(f"DESCRIBE FUNCTION EXTENDED {fn}").collect()
        uc_functions.append({"name": fn, "description": "\n".join(str(r[0]) for r in rows)})
    except Exception as exc:
        uc_functions.append({"name": fn, "description": f"ERROR loading signature: {exc}"})
uc_functions_text = "\n\n".join(f"### {f['name']}\n{f['description']}" for f in uc_functions)
log_event("evidence", "Loaded UC function descriptions", loaded=len(uc_functions), errors=sum(1 for f in uc_functions if f['description'].startswith('ERROR')))

log_event("evidence", "Loading optimized production prompt", prompt_name=PROMPT_NAME)
# Optimized prompt
try:
    prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    optimized_prompt_text = getattr(prompt_obj, "template", None) or prompt_obj.format()
except Exception as exc:
    optimized_prompt_text = f"ERROR loading production prompt: {exc}"
log_event("evidence", "Loaded optimized prompt", chars=len(optimized_prompt_text), load_failed=optimized_prompt_text.startswith('ERROR'))

log_event("evidence", "Loading aligned judge memory", judge_name=ALIGNED_JUDGE_NAME)
# Aligned judge memory. HALO can still run if the local MLflow wheel does not expose mlflow.genai.scorers.
semantic_memory = []
episodic_count = None
try:
    from mlflow.genai.scorers import get_scorer

    aligned_judge = get_scorer(name=ALIGNED_JUDGE_NAME, experiment_id=EXPERIMENT_ID)
    try:
        _ = aligned_judge(
            inputs={"input": [{"role": "user", "content": "How should a hitter approach a pitcher with runners on base?"}]},
            outputs={"response": "Use count, handedness, pitch mix, and location evidence before making a recommendation."},
        )
    except Exception as exc:
        print(f"Judge warmup raised {type(exc).__name__}; continuing with loaded metadata")

    for g in getattr(aligned_judge, "_semantic_memory", []) or []:
        semantic_memory.append({"guideline": g.guideline_text, "source_trace_ids": g.source_trace_ids})
    episodic_count = len(getattr(aligned_judge, "_episodic_memory", []) or [])
except Exception as exc:
    log_event("evidence", "Skipped direct aligned judge memory load", error_type=type(exc).__name__, error=str(exc)[:1000])
    print("HALO will use trace assessments and evaluation metrics as judge evidence instead.")
log_event("evidence", "Aligned judge memory load complete", semantic_guidelines=len(semantic_memory), episodic_examples=episodic_count)

log_event("evidence", "Loading generated skills from UC volume", skills_volume=SKILLS_VOLUME_PATH)
# Generated skills from UC volume
skill_files = []
for path in _list_volume_files(SKILLS_VOLUME_PATH, recursive=True, exclude_prefixes=(HALO_VOLUME_DIR,)):
    if path.endswith((".md", ".txt", ".json")):
        skill_files.append({"path": path, "content": _read_volume_text(path)[:30000]})
log_event("evidence", "Loaded generated skill artifacts", skill_file_count=len(skill_files), skill_paths=[s['path'] for s in skill_files[:10]])

## Build HALO-Compatible Trace JSONL

HALO expects canonical span JSONL. This cell converts MLflow evaluation traces into one root span plus one span per tool call, preserving user query, response, assessments, tool arguments, and tool outputs.

In [ ]:
def _iso_now() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _span(trace_id: str, name: str, attrs: dict[str, Any], parent_span_id: str = "") -> dict[str, Any]:
    sid = uuid.uuid4().hex[:16]
    return {
        "trace_id": trace_id.replace("tr-", "")[:32].ljust(32, "0"),
        "span_id": sid,
        "parent_span_id": parent_span_id,
        "trace_state": "",
        "name": name,
        "kind": "SPAN_KIND_INTERNAL",
        "start_time": _iso_now(),
        "end_time": _iso_now(),
        "status": {"code": "STATUS_CODE_OK", "message": ""},
        "resource": {"attributes": {"service.name": "at-bat-assistant", "project.id": "at-bat-assistant-cais"}},
        "scope": {"name": "mlflow-to-halo", "version": "1"},
        "attributes": attrs,
    }

log_event("traces", "Searching for aligned traces", experiment_id=EXPERIMENT_ID, filter="tag.align = 'use'")
# Prefer the aligned traces if present because these are the examples that drive the harness loop.
traces = mlflow.search_traces(
    locations=[EXPERIMENT_ID],
    filter_string="tag.align = 'use'",
    return_type="list",
)
log_event("traces", "Aligned trace search complete", trace_count=len(traces))
if not traces:
    log_event("traces", "No aligned traces found; falling back to eval-complete traces", filter="tag.eval = 'complete'")
    traces = mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        filter_string="tag.eval = 'complete'",
        return_type="list",
    )
log_event("traces", "Loaded traces for HALO", trace_count=len(traces))

log_event("traces", "Converting MLflow traces to HALO span JSONL")
span_rows = []
trace_summaries = []
for trace in traces:
    tid = trace.info.trace_id
    query = _extract_user_query(trace)
    response = _extract_agent_response(trace)
    tools = _extract_tool_calls(trace)
    assessments = _assessment_summary(trace)
    root = _span(tid, "agent_evaluation_trace", {
        "input.value": query,
        "output.value": response,
        "atbat.trace_id": tid,
        "atbat.user_query": query,
        "atbat.agent_response": response,
        "atbat.assessments": _safe_json(assessments),
        "atbat.tool_call_count": len(tools),
        "atbat.tool_calls": _safe_json(tools),
    })
    span_rows.append(root)
    trace_summaries.append({
        "trace_id": tid,
        "query": query,
        "response_preview": response[:500],
        "tool_count": len(tools),
        "assessments": assessments,
    })
    for i, tool in enumerate(tools, 1):
        status = "STATUS_CODE_ERROR" if str(tool.get("output", "")).lower().startswith("error") else "STATUS_CODE_OK"
        child = _span(tid, f"tool_call_{i}:{tool.get('name')}", {
            "tool.name": tool.get("name"),
            "tool.arguments": tool.get("arguments"),
            "tool.output": str(tool.get("output", ""))[:30000],
            "atbat.trace_id": tid,
        }, parent_span_id=root["span_id"])
        child["status"]["code"] = status
        span_rows.append(child)

with HALO_TRACE_PATH.open("w") as f:
    for row in span_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

log_event("traces", "Wrote HALO span JSONL", span_count=len(span_rows), trace_count=len(trace_summaries), trace_path=str(HALO_TRACE_PATH))
try:
    dbutils.fs.put(f"{HALO_VOLUME_DIR}/halo_traces.in_progress.jsonl", HALO_TRACE_PATH.read_text(encoding="utf-8"), True)
    log_event("traces", "Copied in-progress trace JSONL to UC volume", volume_path=f"{HALO_VOLUME_DIR}/halo_traces.in_progress.jsonl")
except Exception as exc:
    log_event("traces", "Could not copy in-progress trace JSONL", error_type=type(exc).__name__, error=str(exc)[:1000])

## Run HALO

The Databricks endpoint is tried first through OpenAI-compatible routing. If that fails and `OPENAI_API_KEY` is available, the notebook retries against direct OpenAI.

In [ ]:
log_event("halo", "Importing HALO engine modules")
from engine.agents.agent_config import AgentConfig
from engine.engine_config import EngineConfig
from engine.main import run_engine_async
from engine.model_config import ModelConfig
from engine.model_provider_config import ModelProviderConfig
from engine.models.messages import AgentMessage


def _workspace_token() -> str:
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception:
        return w.config.authenticate()


def _make_engine_config(model_name: str, *, provider: ModelProviderConfig | None = None) -> EngineConfig:
    root_turns = int(os.getenv("HALO_ROOT_MAX_TURNS", "8"))
    sub_turns = int(os.getenv("HALO_SUB_MAX_TURNS", "6"))
    max_depth = int(os.getenv("HALO_MAX_DEPTH", "1"))
    max_parallel = int(os.getenv("HALO_MAX_PARALLEL_SUBAGENTS", "2"))
    reasoning_effort = os.getenv("HALO_REASONING_EFFORT", "low")
    root_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=12000)
    sub_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=12000)
    synthesis_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=12000)
    compact_model = ModelConfig(name=model_name, maximum_output_tokens=4000)
    return EngineConfig(
        root_agent=AgentConfig(name="halo-root", model=root_model, maximum_turns=root_turns),
        subagent=AgentConfig(name="halo-sub", model=sub_model, maximum_turns=sub_turns),
        synthesis_model=synthesis_model,
        compaction_model=compact_model,
        model_provider=provider or ModelProviderConfig(),
        maximum_depth=max_depth,
        maximum_parallel_subagents=max_parallel,
    )

log_event("halo", "Building HALO context", trace_count=len(trace_summaries), skill_count=len(skill_files), semantic_guidelines=len(semantic_memory))
halo_context = {
    "uc_functions": uc_functions,
    "optimized_prompt": optimized_prompt_text[:50000],
    "semantic_memory": semantic_memory,
    "episodic_memory_count": episodic_count,
    "skills": skill_files,
    "trace_summaries": trace_summaries,
}

halo_prompt = f"""
You are optimizing the full harness for the Databricks at-bat assistant demo.

Use the trace file as the primary evidence. Also use this harness context:
{_safe_json(halo_context, max_chars=90000)}

Your task:
1. Diagnose recurring harness-level failure modes, not one-off answer mistakes.
2. Rank fixes across prompt, UC tool routing, sufficiency evaluation, Genie fallback, skills, eval data, judge alignment, deployment resources, and observability.
3. Identify which fixes are safe to implement automatically and which need manual review.
4. Produce a Codex-loadable Markdown handoff that another Codex session can read and directly use to suggest prompt and harness changes.
5. Preserve the current demo structure: 00 through 09 remain the existing optimization loop; this HALO pass is an additional whole-harness loop.

The Markdown handoff must use these exact top-level sections:
# Codex Implementation Handoff
## Objective
## Evidence
## Recommended Changes
## File/Notebook Targets
## Patch Plan
## Validation Plan
## Risks and Manual Checks

Reference these notebook targets where relevant:
- notebooks/03_create_agent_definition.ipynb
- notebooks/06-PromptOptimization.ipynb
- notebooks/07-AgentSkillsGeneration.ipynb
- notebooks/08_create_agent_with_skills.ipynb
- notebooks/09-Evaluation.ipynb
- notebooks/10-HALOHarnessOptimization.ipynb

Focus the recommendations on agent prompt changes, tool routing changes, sufficiency/fallback harness changes, Genie fallback instructions, skill-generation inputs, and eval instrumentation. Do not write executable code. Make the handoff concrete enough that Codex can propose patches from it.
"""

log_event("halo", "HALO prompt prepared", prompt_chars=len(halo_prompt), trace_path=str(HALO_TRACE_PATH))

async def _run_halo_with_databricks():
    log_event("halo", "Preparing Databricks model provider", endpoint=HALO_MODEL, base_url=f"{w.config.host.rstrip('/')}/serving-endpoints/")
    provider = ModelProviderConfig(
        base_url=f"{w.config.host.rstrip('/')}/serving-endpoints/",
        api_key=_workspace_token(),
    )
    cfg = _make_engine_config(HALO_MODEL, provider=provider)
    log_event("halo", "Starting HALO run via Databricks", endpoint=HALO_MODEL, reasoning_effort=os.getenv("HALO_REASONING_EFFORT", "low"), root_turns=os.getenv("HALO_ROOT_MAX_TURNS", "8"), sub_turns=os.getenv("HALO_SUB_MAX_TURNS", "6"), max_depth=os.getenv("HALO_MAX_DEPTH", "1"), max_parallel=os.getenv("HALO_MAX_PARALLEL_SUBAGENTS", "2"))
    return await run_engine_async([AgentMessage(role="user", content=halo_prompt)], cfg, HALO_TRACE_PATH, telemetry=False)


async def _run_halo_with_direct_openai():
    log_event("halo", "Preparing direct OpenAI fallback", model=HALO_DIRECT_OPENAI_MODEL)
    if not os.getenv("OPENAI_API_KEY"):
        try:
            openai_secret_scope = os.getenv("OPENAI_SECRET_SCOPE", "<your-secret-scope>")
            os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope=openai_secret_scope, key="openai_api_key")
        except Exception:
            pass
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Databricks HALO route failed and OPENAI_API_KEY is not available for direct OpenAI fallback")
    cfg = _make_engine_config(HALO_DIRECT_OPENAI_MODEL)
    log_event("halo", "Starting HALO run via direct OpenAI", model=HALO_DIRECT_OPENAI_MODEL)
    return await run_engine_async([AgentMessage(role="user", content=halo_prompt)], cfg, HALO_TRACE_PATH, telemetry=False)

start = time.time()
try:
    log_event("halo", "Running HALO through Databricks OpenAI-compatible endpoint")
    halo_items = await _run_halo_with_databricks()
    halo_route = "databricks"
    log_event("halo", "Databricks HALO route completed", output_items=len(halo_items))
except Exception as databricks_exc:
    log_event("halo", "Databricks HALO route failed; trying direct OpenAI fallback", error_type=type(databricks_exc).__name__, error=str(databricks_exc)[:2000])
    print(type(databricks_exc).__name__, str(databricks_exc)[:2000])
    try:
        halo_items = await _run_halo_with_direct_openai()
        halo_route = "direct_openai"
        log_event("halo", "Direct OpenAI HALO route completed", output_items=len(halo_items))
    except Exception as openai_exc:
        halo_route = "failed"
        halo_items = []
        raise RuntimeError(
            "HALO failed through both Databricks and direct OpenAI. "
            f"Databricks error: {databricks_exc}\nDirect OpenAI error: {openai_exc}"
        ) from openai_exc

elapsed = time.time() - start
log_event("halo", "HALO run complete", route=halo_route, elapsed_seconds=round(elapsed, 1), output_items=len(halo_items))

In [ ]:
def _item_to_dict(item):
    try:
        return item.model_dump(mode="json")
    except Exception:
        return {"repr": repr(item)}

halo_payload = {
    "route": halo_route,
    "model": HALO_MODEL if halo_route == "databricks" else HALO_DIRECT_OPENAI_MODEL,
    "elapsed_seconds": elapsed,
    "trace_path": str(HALO_TRACE_PATH),
    "items": [_item_to_dict(item) for item in halo_items],
}
HALO_OUTPUT_JSON.write_text(json.dumps(halo_payload, indent=2, default=str), encoding="utf-8")
log_event("artifacts", "Wrote HALO JSON payload locally", path=str(HALO_OUTPUT_JSON), item_count=len(halo_items))

texts = []
for item in halo_items:
    data = _item_to_dict(item)
    obj = data.get("item", {}) if isinstance(data, dict) else {}
    if isinstance(obj, dict) and obj.get("role") == "assistant":
        content = obj.get("content")
        if isinstance(content, str):
            texts.append(content)
        elif isinstance(content, list):
            for part in content:
                if isinstance(part, dict) and part.get("text"):
                    texts.append(part["text"])

handoff = "\n\n".join(texts).strip()
if not handoff:
    handoff = json.dumps(halo_payload, indent=2, default=str)[:100000]

header = f"""# HALO Harness Optimization Handoff

Generated: {datetime.now(timezone.utc).isoformat()}
Route: {halo_route}
Model: {HALO_MODEL if halo_route == 'databricks' else HALO_DIRECT_OPENAI_MODEL}
Trace count: {len(traces)}
Span count: {len(span_rows)}

"""
HALO_HANDOFF_MD.write_text(header + handoff + "\n", encoding="utf-8")
log_event("artifacts", "Wrote Codex handoff locally", path=str(HALO_HANDOFF_MD), handoff_chars=len(handoff))
display(Markdown(HALO_HANDOFF_MD.read_text()[:20000]))

In [ ]:
# Persist HALO artifacts to the same UC volume family used by skills.
halo_volume_dir = HALO_VOLUME_DIR
try:
    log_event("artifacts", "Persisting final HALO artifacts to UC volume", volume_dir=halo_volume_dir)
    dbutils.fs.mkdirs(halo_volume_dir)
    dbutils.fs.put(f"{halo_volume_dir}/halo_traces.jsonl", HALO_TRACE_PATH.read_text(encoding="utf-8"), True)
    dbutils.fs.put(f"{halo_volume_dir}/halo_output.json", HALO_OUTPUT_JSON.read_text(encoding="utf-8"), True)
    dbutils.fs.put(f"{halo_volume_dir}/codex_handoff.md", HALO_HANDOFF_MD.read_text(encoding="utf-8"), True)
    log_event("artifacts", "Persisted final HALO artifacts to UC volume", volume_dir=halo_volume_dir)
except Exception as exc:
    log_event("artifacts", "Could not persist final HALO artifacts", error_type=type(exc).__name__, error=str(exc)[:1000])

codex_handoff_markdown = HALO_HANDOFF_MD.read_text(encoding="utf-8")
log_event("done", "HALO notebook completed; displaying full codex_handoff.md", handoff_chars=len(codex_handoff_markdown))
display(Markdown(codex_handoff_markdown))


# HALO Harness Optimization Handoff

Generated: 2026-05-18T02:43:34.308915+00:00
Route: databricks
Model: gpt-5-4-external
Trace count: 19
Span count: 124

## Dataset quick read (what we can prove)
- Total traces: **19**; traces with errors: **8**
- Dataset raw size: **~294 KB** (so we can inspect individual traces directly)
- Service concentrated on: **`at-bat-assistant`**

Below are recurring *harness-level* failure modes I observed in the provided sample traces, with concrete span/tool evidence.

---

## Failure mode 1 — Databricks function routing/availability issues (timeouts + 504s)
**Symptom:** The assistant calls Databricks-hosted MCP functions and repeatedly gets gateway timeouts and/or 90s execution timeouts. This blocks retrieval of the core matchup/tendency data and produces degraded or empty outputs.

**Concrete evidence**
1) **Trace `2ab055d35560d66491f51afaa55eeeb3` (error)**
- Span: `tool_call_1:main__at_bat_assistant__lookup_player_by_name`
  - Error message in `atbat.tool_calls.output`:
  - **“Error: Execution timed out after 90 seconds. Provisioning resources for function execution may be taking longer than expected.”**
- Span: `tool_call_2:main__at_bat_assistant__lookup_player_by_name`
  - Error:
  - **“Server error '504 Gateway Timeout' … https://dbc-.../api/2.0/mcp/functions/main/at_bat_assistant”**
- Span: `tool_call_4:main__at_bat_assistant__get_pitcher_tendency_with_runners`
  - Error:
  - **“Error: BAD_REQUEST: Missing parameter value for b”**
- Span: `tool_call_6:main__at_bat_assistant__get_pitcher_tendency_with_runners`
  - Error:
  - **90s execution timeout** (same provisioning-timeout pattern)

2) **Trace `2ca95b7ae4969fdec1302cd2892292c1` (error)**
- Span: `tool_call_4:main__at_bat_assistant__pitcher_arsenal_lookup`
  - Error: **“Server error '504 Gateway Timeout' … at_bat_assistant'”**
- Span: `tool_call_5:main__at_bat_assistant__get_pitcher_tendency_by_count`
  - Error: **90s execution timeout** (provisioning resources)
- Span: `tool_call_7:main__at_bat_assistant__get_pitcher_tendency_by_count`
  - Error: **504 Gateway Timeout**

**Impact on demo:** core retrieval steps intermittently fail, so the assistant either (a) can’t compute tendencies or (b) proceeds with partial/empty tool results, leading to weak or non-actionable recommendations.

---

## Failure mode 2 — Tool input schema / parameter completeness bugs (“Missing parameter value …”)
**Symptom:** Even when routing reaches the right tool, the harness/model sometimes omits required parameters. This produces deterministic **BAD_REQUEST** errors that the assistant doesn’t correct automatically.

**Concrete evidence**
- **Trace `2ab055d35560d66491f51afaa55eeeb3`**
  - Span: `tool_call_4:main__at_bat_assistant__get_pitcher_tendency_with_runners`
    - **“BAD_REQUEST: Missing parameter value for b”**
  - Span: `tool_call_5:main__at_bat_assistant__get_pitcher_tendency_with_runners`
    - **“BAD_REQUEST: Missing parameter value for p_on_2b”**
- **Trace `c7afbaa7af838eafde79aabf40b239a8`**
  - Span: `tool_call_4:main__at_bat_assistant__get_pitcher_tendency_by_count`
    - **“BAD_REQUEST: Missing parameter value for b”**
- **Trace `c3a3ccc8e293db06aaabdc16766aa325`**
  - Span: `tool_call_6:main__at_bat_assistant__recommend_batter_matchups_by_team`
    - **“BAD_REQUEST: Missing parameter value for season_year”**
  - Span: `tool_call_8:main__at_bat_assistant__get_team_batters`
    - **“BAD_REQUEST: Missing parameter value for season_year”**
  - (Additionally, tool timeouts occur around these)

**Impact on demo:** harness/tool-calling layer generates invalid tool arguments; without robust retry/fill-in defaults, the system falls back to generic LLM text.

---

## Failure mode 3 — Retrieval insufficiency / query returns empty, but no effective fallback
**Symptom:** Even when tool calls succeed, the system may receive **EMPTY** outputs (e.g., `genie_space_query` returns EMPTY) or empty DataFrame rows, and the assistant doesn’t recover with alternate sources/strategies.

**Concrete evidence**
- **Trace `2ab055d35560d66491f51afaa55eeeb3` (ends with empty retrieval)**
  - Span: `tool_call_13:genie_space_query`
    - Tool output: **`EMPTY`**
  - Agent output: apologetic “no available information…”
- **Trace `c3a3ccc8e293db06aaabdc16766aa325`**
  - Span: `tool_call_15:genie_space_query`
    - Tool output: **`EMPTY`**
  - Agent output explicitly says there’s **no data** and provides no alternative strategy.
- **Trace `c3a3ccc8e293db06aaabdc16766aa325`**
  - Span: `tool_call_3:main__at_bat_assistant__recommend_batter_matchups_by_team` succeeded once but returned:
    - `rows: []` (no recommendations)
- **Trace `2ca95b7ae4969fdec1302cd2892292c1`**
  - Span: `tool_call_9:genie_space_query`
    - Output table shows `matchup_count = 0` and totals are zero.

**Impact on demo:** “no data” responses occur frequently and appear to be treated as acceptable end states rather than triggers to broaden the query (e.g., different seasons, players, or data sources).

---

## Failure mode 4 — Sufficiency evaluation / fallback logic failures (LLM_JUDGE signals don’t prevent bad outcomes)
**Symptom:** The evaluation/judge step can mark responses as relevant enough, yet the response remains non-actionable due to tool failures/empty results. Also, “fallback” seems to degrade into generic apology or “no recommendations” rather than a structured alternative.

**Concrete evidence (judge vs outcome mismatch)**
- **Trace `2ab055d35560d66491f51afaa55eeeb3`**
  - Core tool failures: timeouts/504 + BAD_REQUESTs (see Failure modes 1–2)
  - Agent output: **apology / no information**
  - Assessments include:
    - `baseball_analysis_base` (LLM_JUDGE) value **1.0** with rationale: assistant “fails to analyze any available data… no actionable recommendation”
- **Trace `c3a3ccc8e293db06aaabdc16766aa325`**
  - Multiple tool issues (time out, BAD_REQUEST for `season_year`)
  - Final answer: “there is no data… no specific batters…”
  - The evaluation includes `relevance_to_query` = **yes** (meaning it addressed the question), but it still fails the “actionable alternative” requirement.
- **Trace `2ab055d35560d66491f51afaa55eeeb3`**
  - `baseball_analysis_base` includes human rationale: **“Tools were erroring out and request took forever”** (i.e., the harness state is acknowledged, but the system still outputs a non-recovery response).

**Impact on demo:** evaluation doesn’t trigger a robust recovery plan (e.g., retry strategy for databricks calls, parameter repair, alternative querying) once the system learns tool retrieval is insufficient.

---

## Failure mode 5 — Partial progress without end-to-end gating (continues despite tool errors/timeouts)
**Symptom:** The orchestration appears to keep going even after multiple tool calls fail, rather than aborting/retrying with stricter gating. This leads to “hybrid” states: some tools succeed, others fail, and the final answer may be derived from incomplete data.

**Concrete evidence**
- **Trace `2ab055d35560d66491f51afaa55eeeb3`**
  - Player lookup: 1st attempt timed out, 2nd got 504, 3rd succeeded
  - Tendency-with-runners: multiple attempts fail (BAD_REQUESTs + timeouts), yet later steps proceed (pitcher arsenal lookup succeeds; then `genie_space_query` returns EMPTY)
- **Trace `c3a3ccc8e293db06aaabdc16766aa325`**
  - Recommendation tool timed out once, then later succeeds but returns empty; other tools have BAD_REQUEST missing `season_year`.
  - Despite that, the pipeline still attempts follow-on matchup retrieval and then ends with EMPTY genie query.

---

# Categorized failure modes (summary)
1) **Databricks MCP availability / provisioning timeouts**
   - 90s provisioning timeout; **504 Gateway Timeout**
2) **Harness parameter/schema defects**
   - BAD_REQUEST: missing `b`, `p_on_2b`, `season_year`
3) **Insufficient retrieval leading to empty/“no data” outcomes**
   - `genie_space_query` returns `EMPTY`; recommendation tables `rows: []`
4) **Fallback + sufficiency evaluation not preventing poor end states**
   - Judge may mark “relevant” while response remains non-actionable/no-recovery
5) **No strict orchestration gating after repeated tool failure**
   - Continues with partial data; final answer degrades

---

If you want, I can run a regex scan across *all error traces* to quantify which error strings (e.g., `504 Gateway Timeout`, `Execution timed out after 90 seconds`, `BAD_REQUEST: Missing parameter value`) appear most and which tool names they cluster around.

Below is a **ranked set of likely harness-level fixes** inferred from the traces (notably: **tool timeouts/504**, **BAD_REQUEST missing parameters**, **Genie returning EMPTY**, and **evaluation/judge marking “bad” for non-actionable or wrong-handling outputs**).  

> Assumption: project is an “at-bat-assistant” style agent with tools like `main__at_bat_assistant__get_pitcher_tendency_with_runners`, `recommend_batter_matchups_by_team`, and a `genie_space_query` fallback, plus a sufficiency/eval pipeline using `LLM_JUDGE`.

## 1) UC tool routing: add “parameter completion + validation” before calling tools (prevents BAD_REQUEST)
**What we saw:** `get_pitcher_tendency_with_runners` fails with `BAD_REQUEST: Missing parameter value for b` / `p_on_2b` / `season_year`.  
**Fix:** In your UC/tool router (or tool wrapper), enforce a **schema/required-fields map** for each tool; if missing, either:
- derive from context (e.g., map bases occupancy booleans → expected numeric params), or
- ask the agent for the missing values explicitly (manual review gate if ambiguous), or
- fall back to a reduced query variant that doesn’t require the missing field.

**Safe for auto-implement vs manual review**
- **Auto-implement (safe):** strict validation + deterministic param derivation + failing fast with a structured “need input X” signal.
- **Manual review:** only for cases where the derivation is ambiguous (e.g., interpreting “right-handed batters” vs `b_hand` mapping rules).

**File/notebook targets (typical)**
- `src/<...>/tool_router*.py` (UC routing layer)
- `src/<...>/tool_wrappers/*.py` (input normalization per tool)
- `notebooks/uc_routing_sanity_checks.ipynb` (add unit tests for BAD_REQUEST cases)

---

## 2) Deployment/resources: reduce tool timeout sensitivity + add retry/backoff for transient 504/timeout
**What we saw:** repeated `Execution timed out after 90 seconds` and `504 Gateway Timeout` from Databricks MCP endpoint during `lookup_player_by_name` and others.  
**Fix:**
- Add **short-circuit retry policy** for transient failures (e.g., 504/timeout) with exponential backoff.
- Add **circuit breaker** so the agent doesn’t repeatedly hammer the same failing tool.
- Consider **lower concurrency / batch smaller queries** if the tools support it.
- Increase server-side limits (if you control the MCP/Databricks endpoint) or tune query execution.

**Safe for auto-implement vs manual review**
- **Auto-implement (safe):** retry/backoff + circuit breaker + request cancellation.
- **Manual review:** only if changing infra knobs (timeouts, worker counts) needs environment coordination.

**File/notebook targets**
- `src/<...>/tool_client*.py` (HTTP/MCP client)
- `src/<...>/retry_policy*.py`
- `deploy/k8s/values*.yaml` or `terraform/*` (worker/timeouts), if applicable
- `notebooks/load_test_tool_timeouts.ipynb`

---

## 3) Genie fallback: fix “EMPTY but still judged as insufficient” by adding a structured alternative strategy
**What we saw:** `genie_space_query` returns `EMPTY`, yet the agent sometimes ends with “no data” statements without an alternative plan (and judges penalize lack of actionable advice).  
**Fix:** When Genie returns EMPTY, route to one of:
- **Data sufficiency expansion**: query adjacent seasons/nearby stat windows (if allowed by policy).
- **Different query templates**: e.g., ask for “pitcher tendencies by runner state” separately, not the full combined question.
- **Last-resort reasoning template**: provide an explicit “what we checked / what we couldn’t fetch / what you can provide” plus a coaching-oriented guidance (still grounded, but not “nothing at all”).

**Safe for auto-implement vs manual review**
- **Auto-implement (safe):** control-flow logic for EMPTY detection + selecting an alternative template.
- **Manual review:** the exact coaching/wording policy for the fallback response.

**File/notebook targets**
- `src/<...>/genie_fallback*.py` or wherever `genie_space_query` is called
- `src/<...>/response_templates/*.md` (or `.txt`)
- `notebooks/fallback_regression_tests.ipynb`

---

## 4) Prompt: enforce “if tools fail/empty, do an alternative action plan” (not just “no info”)
**What we saw:** outputs like “I’m sorry, but I don’t have any available information…” and “no recommendations” get judged as unhelpful/non-actionable.  
**Fix:** Add explicit prompt constraints:
- Always include: **(a)** attempted data sources, **(b)** failure modes encountered (timeout/empty), **(c)** next-best action (what to ask next or what to broaden), **(d)** a coaching-style “decision framework” even when exact matchup numbers are missing.

**Safe for auto-implement vs manual review**
- **Manual review recommended:** prompt changes affect quality/format; validate with offline eval first.

**File/notebook targets**
- `prompts/agent_system_prompt*.txt` (or similar)
- `prompts/coach_response_template*.md`
- `notebooks/prompt_ablation_sufficiency.ipynb`

---

## 5) Sufficiency evaluation: correct the “sufficiency gate” so it doesn’t prematurely stop after partial/empty tool results
**What we saw:** “no data” answers that are factually consistent but fail the harness’s *actionability* expectation; judges marked low `baseball_analysis_base` / `baseball_language`.  
**Fix:** Update sufficiency evaluation to check for:
- **presence of actionable content** (at least one recommended batter/pitching plan *or* an explicit next-step plan when data absent),
- **presence of baseball-specific terminology** (or an enforced “terminology checklist”),
- **grounding traceability** (references to which tool outputs were empty/timed out).

Also, ensure the agent doesn’t interpret “tool returned empty” as “task complete”.

**Safe for auto-implement vs manual review**
- **Auto-implement (safe):** add new rubric signals + thresholds; rewire sufficiency gate conditions.
- **Manual review:** calibration of judge thresholds per business requirement.

**File/notebook targets**
- `src/<...>/eval/sufficiency_*.py`
- `src/<...>/eval/rubrics/*.json` (or YAML)
- `notebooks/sufficiency_threshold_calibration.ipynb`

---

## 6) Judge alignment: reduce false negatives/positives by matching rubric to desired output contract
**What we saw:** LLM judge penalizes responses that aren’t “insightful/strategic” and that lack baseball terminology; another marks “no” when the answer doesn’t provide what the user asked even if the tool was empty.  
**Fix:**
- Align judge rubric to the **contract**: e.g., “When matchup data is missing, the required fallback is an alternative strategy, not refusal.”
- Make the judge explicitly check for fallback compliance (next actions + framework).
- Add “grading invariants” to prevent the judge from ignoring known failure modes (timeouts/EMPTY).

**Safe for auto-implement vs manual review**
- **Manual review recommended:** judge prompts/rubrics are easy to overfit; validate against a labeled set.

**File/notebook targets**
- `src/<...>/eval/judge_prompts/*.txt`
- `src/<...>/eval/judge_rubrics/*.json`
- `notebooks/judge_alignment_review.ipynb`

---

## 7) UC routing: add “tool fallback ladder” + reduced-parameter fallbacks (avoid repeated failing calls)
**What we saw:** repeated calls to `get_pitcher_tendency_with_runners` with varying params, still triggering timeouts or missing fields.  
**Fix:** Implement a deterministic **ladder** per user intent:
1) exact tool call  
2) if tool fails → retry with reduced payload / fewer runner/base parameters  
3) if empty → Genie/other retrieval  
4) if still empty → response template requiring next-step plan

**Safe for auto-implement vs manual review**
- **Auto-implement (safe):** routing control-flow + ladder ordering.
- **Manual review:** ladder order / which reduced calls are allowed.

**File/notebook targets**
- `src/<...>/uc_router/*.py`
- `src/<...>/tool_fallback_ladders*.py`
- `notebooks/routing_ladder_simulations.ipynb`

---

## 8) Skills: add a “Baseball terminology + coaching framing” skill used whenever fallback/insufficient-data happens
**What we saw:** judges penalized `baseball_language` due to missing baseball terminology and/or too generic phrasing.  
**Fix:** Create a small deterministic “skills module” that:
- injects a terminology checklist (e.g., pitch locations, pitch types, handedness, count strategy, runner state),
- frames the output as a coaching recommendation *or* a decision framework when exact data is unavailable.

**Safe for auto-implement vs manual review**
- **Auto-implement (safe):** terminology injection + structured response sections.
- **Manual review:** terminology appropriateness for your domain constraints.

**File/notebook targets**
- `src/<...>/skills/baseball_language_skill*.py`
- `src/<...>/skills/fallback_framework_skill*.py`
- `notebooks/skill_terminology_coverage.ipynb`

---

## 9) Eval data: add targeted regression cases for EMPTY/timeout/missing-param and judge expectations
**What we saw:** multiple failure modes in only 19 traces; you need eval coverage.  
**Fix:** Add synthetic + captured harness cases:
- “tool timeout 504 then Genie EMPTY”
- “BAD_REQUEST missing season_year”
- “correct tool outputs but agent returns refusal”
- “correct data exists but agent under-analyzes”
- “language missing during fallback”

**Safe for auto-implement vs manual review**
- **Auto-implement (safe):** adding eval examples + harness scripts.
- **Manual review:** labeling/expected response contract for “fallback strategy” outputs.

**File/notebook targets**
- `eval_datasets/<...>/*.jsonl` (or `.csv`)
- `notebooks/build_eval_set_from_traces.ipynb`
- `notebooks/harness_regression_suite.ipynb`

---

## 10) Observability: add structured error + “attempted tools” fields into traces/logs
**What we saw:** we can read tool errors in trace payloads, but harness fixes will be faster with explicit structured fields (e.g., `tool_failure_reason`, `sufficiency_gate_result`, `fallback_used=true`).  
**Fix:**
- Emit standardized span attributes for: failure type (timeout/504/bad_request/empty_result), which routing ladder step was used, and sufficiency gate outcome.
- Add dashboards/alerts for top tool failures + rates.

**Safe for auto-implement vs manual review**
- **Auto-implement (safe):** instrumentation + dashboards.
- **Manual review:** choosing alert thresholds.

**File/notebook targets**
- `src/<...>/telemetry/*.py` (span attribute enrichment)
- `src/<...>/logging/*.py`
- `observability/grafana/*.json` or `dashboards/*.yaml`

---

### Quick “Top 3” to implement first (highest leverage)
1) **Tool routing param validation/completion** (fixes BAD_REQUEST)  
2) **Retry/backoff + circuit breaker for 504/timeout** (fixes provisioning/resources failure mode)  
3) **EMPTY fallback ladder + action plan response template** (fixes non-actionable judge penalties)

If you share the **repo structure** (or the actual filenames/paths for the UC router, tool client, judge/sufficiency, and genie fallback), I can convert the “typical targets” above into an exact patch plan.

# Codex Implementation Handoff
## Objective
Optimize the full “Databricks at-bat assistant” harness to reduce **recurring harness-level failures** observed in the trace dataset, focusing on:
- **Prompt + tool routing** correctness (avoid BAD_REQUEST/missing parameters)
- **Robust fallback ladder** when Databricks tools time out or return **EMPTY**
- **Sufficiency evaluation + Genie fallback** so the system produces **actionable** outcomes rather than “no data / apologies”
- **Skills + eval instrumentation** to prevent regressions and improve judge alignment

## Evidence
Dataset scope (from traces):
- `total_traces`: **19**
- `error_trace_count`: **8**
- `service_names`: **at-bat-assistant**
- `raw_jsonl_bytes`: **294,202 (~294 KB)** → trace-level inspection feasible.

Key recurring evidence (representative traces):
1) **Databricks tool availability/timeouts (504 + 90s provisioning timeout)**
- **Trace `2ab055d35560d66491f51afaa55eeeb3`**
  - `tool_call_1/2`: `main__at_bat_assistant__lookup_player_by_name`  
    - “**Execution timed out after 90 seconds. Provisioning resources for function execution may be taking longer than expected.**”
    - “**Server error '504 Gateway Timeout' … /api/2.0/mcp/functions/main/at_bat_assistant**”
  - `tool_call_4/6`: `main__at_bat_assistant__get_pitcher_tendency_with_runners`
    - “**BAD_REQUEST: Missing parameter value for b**”
    - “**90s execution timeout**”
- **Trace `2ca95b7ae4969fdec1302cd2892292c1`**
  - `tool_call_4`: `main__at_bat_assistant__pitcher_arsenal_lookup`
    - “**504 Gateway Timeout**”
  - `tool_call_5`: `main__at_bat_assistant__get_pitcher_tendency_by_count`
    - “**90s execution timeout**”
    - “**504 Gateway Timeout**”

2) **Tool input/schema completeness bugs (BAD_REQUEST missing parameters)**
- **Trace `2ab055d35560d66491f51afaa55eeeb3`**
  - `get_pitcher_tendency_with_runners`:  
    - “**BAD_REQUEST: Missing parameter value for b**”
    - “**BAD_REQUEST: Missing parameter value for p_on_2b**”
- **Trace `c7afbaa7af838eafde79aabf40b239a8`**
  - `get_pitcher_tendency_by_count`:
    - “**BAD_REQUEST: Missing parameter value for b**”
- **Trace `c3a3ccc8e293db06aaabdc16766aa325`**
  - `recommend_batter_matchups_by_team`:
    - “**BAD_REQUEST: Missing parameter value for season_year**”
  - `get_team_batters`:
    - “**BAD_REQUEST: Missing parameter value for season_year**”

3) **Retrieval insufficiency → Genie returns EMPTY, but fallback doesn’t produce actionable alternatives**
- **Trace `2ab055d35560d66491f51afaa55eeeb3`**
  - `tool_call_13`: `genie_space_query` → tool output **`EMPTY`**
  - Agent output ends in an **apology / no available information** state.
- **Trace `c3a3ccc8e293db06aaabdc16766aa325`**
  - `tool_call_15`: `genie_space_query` → **`EMPTY`**
  - Agent ends with “there is no data …” without a structured “next-best plan”.

4) **Sufficiency evaluation / judge alignment does not prevent poor end states**
- Judging/rubric accepts “relevance” even when outputs are non-actionable (e.g., timeouts + EMPTY + final “no data”).
- Trace `2ab055d35560d66491f51afaa55eeeb3` includes judge rationale indicating the system “acknowledged tool errors and request took forever” yet still failed to recover into a useful fallback.

5) **No strict orchestration gating: continues after repeated tool failures**
- System proceeds to later steps even after:
  - parameter BAD_REQUEST
  - timeouts/504
  - partial success
- This creates hybrid “some data, then EMPTY, then apology” outcomes.

## Recommended Changes
Ranked by leverage and recurrence risk. Each item includes **automatic-safe vs manual-review**.

### A. Tool routing: add required-parameter validation + completion (highest priority)
**Problem solved:** BAD_REQUEST missing parameters for `b`, `p_on_2b`, `season_year`.
**Change:**
- Add a UC/tool-router “**schema contract**” per tool:
  - Required fields list (from UC tool definitions)
  - Type checks
- If missing:
  - **Auto-derive** from conversational/context signals where unambiguous (e.g., “base occupancy” → booleans)
  - If ambiguous → **explicitly request the missing input** (or select a safe reduced query variant that doesn’t require it)
- Fail fast with a structured error object so the harness chooses fallback ladder instead of letting the LLM stumble.

**Safe for auto-implement:** ✅  
**Manual review needed:** only for ambiguous derivations (handedness/stand mapping edge cases).

**Patch targets:**
- `notebooks/06-PromptOptimization.ipynb` (prompt/tool-calling constraints & “request missing params” behavior)
- `notebooks/10-HALOHarnessOptimization.ipynb` (harness gating rules)
- (Repo code path where UC routing is implemented; if not in notebooks, adjust the router module used by the harness)

---

### B. Tool reliability: retry/backoff + circuit breaker for 504/timeout
**Problem solved:** “Execution timed out after 90 seconds” + “504 Gateway Timeout”.
**Change:**
- In the tool client / harness wrapper:
  - Retry policy for **transient** errors (`504`, execution timeout)
  - Exponential backoff with jitter
  - Circuit breaker: after N failures, skip that tool for the current request (avoid hammering)
- Ensure retries do not cause parameter drift (tie retries to the same validated param set).

**Safe for auto-implement:** ✅  
**Manual review needed:** infra knobs (timeouts, limits) if any.

**Patch targets:**
- `notebooks/10-HALOHarnessOptimization.ipynb` (retry/fallback policy)
- tool client module used by the MCP/Databricks tool calls

---

### C. Genie fallback ladder: replace “EMPTY → apology” with structured action plan
**Problem solved:** `genie_space_query` returns **EMPTY**, ending in non-actionable refusal/apology.
**Change:**
- Implement a deterministic **fallback ladder**:
  1) Exact tool query (validated params)
  2) Reduced/alternate tool template (e.g., query by count without runner state if runner-state tool fails)
  3) Genie search with a **more specific query template** (or alternate template)
  4) If still EMPTY: respond using a **required fallback response contract**:
     - “What I tried”
     - “What failed (timeout/empty)”
     - “Next-best action” (e.g., ask for missing parameters, or propose generic strategy framework that’s still helpful)
- The fallback response must always include a coaching-style alternative (not just “no data”).

**Safe for auto-implement:** ✅  
**Manual review needed:** fallback wording policy and whether to allow “generic coaching” under your product requirements.

**Patch targets:**
- Genie fallback module/instructions used by the harness
- `notebooks/10-HALOHarnessOptimization.ipynb`
- `notebooks/09-Evaluation.ipynb` (eval cases must include EMPTY scenarios)

---

### D. Prompt contract: force “alternative action plan” on tool failure/EMPTY
**Problem solved:** prompt allows terminal “no info” outcomes.
**Change:**
Add explicit instruction to the agent prompt:
- If any critical tool fails (timeout/504) **or** returns EMPTY:
  - do not end with apology
  - execute fallback ladder steps
  - if final data absent, produce an **actionable decision framework** + ask for missing inputs

**Safe for auto-implement:** ⚠️ (prompt changes can affect quality)  
**Manual review needed:** ✅ (review prompt text + run eval)

**Patch targets:**
- `notebooks/06-PromptOptimization.ipynb`
- agent/system prompt used in `main.at_bat_assistant.atbat_assistant_prompt`

---

### E. Sufficiency evaluation: add “actionability” requirements for terminal states
**Problem solved:** judge/sufficiency doesn’t trigger recovery even when response is non-actionable.
**Change:**
- Update sufficiency rubric to distinguish:
  - “relevant but unhelpful due to tool failures”
  - “unhelpful refusal/no alternative plan”
- New pass/fail criteria:
  - If no matchup/tendency data is present, response must still contain:
    - at least one actionable next step (ask for missing inputs OR provide decision framework)
    - explicit mention of attempted retrieval and/or reason data is missing

**Safe for auto-implement:** ✅ (logic/rubric change is deterministic)  
**Manual review needed:** judge thresholds calibration.

**Patch targets:**
- `notebooks/09-Evaluation.ipynb`
- `notebooks/10-HALOHarnessOptimization.ipynb`

---

### F. Judge alignment: explicitly score fallback compliance (not just “relevance”)
**Problem solved:** LLM_JUDGE marks relevance but fails to penalize missing fallback structure.
**Change:**
- Update judge prompt/rubric to include a “fallback contract checklist”:
  - tried tools / observed empty
  - next-best action present
  - baseball/coaching framing present
- Prevent judge from awarding high scores for “no data apologies”.

**Safe for auto-implement:** ⚠️  
**Manual review needed:** ✅ (rubric calibration + sample review)

**Patch targets:**
- judge prompt/rubric used by `baseball_analysis_base` (wherever it’s configured)
- `notebooks/09-Evaluation.ipynb`

---

### G. Skills: add/adjust a “fallback baseball coaching framework” skill
**Problem solved:** language/terminology/coaching framing deficiencies during fallback.
**Change:**
- Introduce a deterministic skill used when data is missing:
  - structured sections (what we tried → why missing → coaching framework)
  - includes baseball terminology (pitch types, zones, handedness, count strategy)
- Use this skill only when fallback ladder triggers.

**Safe for auto-implement:** ✅  
**Manual review needed:** domain-appropriate coaching style

**Patch targets:**
- `notebooks/07-AgentSkillsGeneration.ipynb`
- `notebooks/08_create_agent_with_skills.ipynb`

---

### H. Eval data: add regression cases for the exact failure motifs
**Problem solved:** small trace count (19) makes it easy to overfit; missing coverage for EMPTY/timeout/BAD_REQUEST.
**Change:**
Add targeted eval set entries for:
- timeout/504 then Genie EMPTY
- BAD_REQUEST missing parameter then recovery via router completion or reduced query
- Genie EMPTY with required fallback response contract
- “partial tool success” hybrid states

**Safe for auto-implement:** ✅  
**Manual review needed:** labeling/expected contract for fallback outputs

**Patch targets:**
- `notebooks/09-Evaluation.ipynb`
- `notebooks/10-HALOHarnessOptimization.ipynb`

---

### I. Observability: instrument failure taxonomy and fallback ladder step
**Problem solved:** slower iteration without structured telemetry.
**Change:**
- Add standardized span attributes/events:
  - failure_type: `timeout_90s`, `http_504`, `bad_request_missing_param`, `genie_empty`
  - attempted_tools: list
  - fallback_ladder_step_used
  - sufficiency_gate_result
- Use these in dashboards and in eval explanations.

**Safe for auto-implement:** ✅  
**Manual review needed:** thresholds/dashboards setup

**Patch targets:**
- tool wrapper / telemetry module
- `notebooks/10-HALOHarnessOptimization.ipynb`

## File/Notebook Targets
Primary:
- `notebooks/06-PromptOptimization.ipynb` (prompt constraints for failure/empty handling)
- `notebooks/07-AgentSkillsGeneration.ipynb` (generate/update fallback coaching skill)
- `notebooks/08_create_agent_with_skills.ipynb` (attach skills to the agent with correct routing)
- `notebooks/09-Evaluation.ipynb` (update sufficiency/judge rubric + add eval cases)
- `notebooks/10-HALOHarnessOptimization.ipynb` (fallback ladder orchestration + retry policies + gating)
Supporting:
- `notebooks/03_create_agent_definition.ipynb` (if agent/system prompt wiring changes needed)

## Patch Plan
1) **Implement tool parameter validation/completion**
   - Add per-tool required-field checks in UC router
   - Auto-derive unambiguous missing values
   - If cannot derive → request missing parameters or choose reduced tool variant
   - Ensure INVALID_TOOL_ARGS triggers fallback ladder immediately
2) **Harden tool calls for 504/timeouts**
   - Add retry/backoff only for transient errors
   - Add circuit breaker per tool per request
   - Cap total attempts so orchestration time remains bounded
3) **Build fallback ladder for EMPTY/failed retrieval**
   - Ensure `genie_space_query == EMPTY` triggers:
     - alternate templates or reduced queries
     - and finally a **fallback response contract** (actionable coaching/framework)
4) **Update prompt + skills for fallback contract compliance**
   - Prompt: never end with apology/no data without next-best action
   - Skills: add structured baseball coaching fallback module
5) **Update sufficiency evaluation + judge alignment**
   - Add actionability checks
   - Add fallback contract checklist scoring
6) **Add eval regression suite mirroring trace motifs**
   - timeouts/504 + EMPTY
   - BAD_REQUEST missing params + recovery
   - partial tool success hybrid
7) **Add observability fields**
   - failure taxonomy + attempted_tools + ladder step

## Validation Plan
- Run existing evaluation suite from `notebooks/09-Evaluation.ipynb`.
- Add new test cases explicitly covering:
  - `504 Gateway Timeout` sequences
  - `Execution timed out after 90 seconds` sequences
  - `BAD_REQUEST: Missing parameter value for ...`
  - Genie returning `EMPTY`
- Success criteria (per request):
  - **No BAD_REQUEST** should reach final response (either corrected or handled via fallback ladder)
  - For EMPTY outcomes, final response must:
    - include “what we tried”
    - include a structured next-best action/coaching framework
- Compare:
  - before/after pass rate on “actionability” metrics
  - reduction in terminal apology/refusal states
  - reduction in timeouts reaching the final answer

## Risks and Manual Checks
- **Manual review required (prompt/judge/skill wording):**
  - Ensure fallback coaching remains within demo expectations (product/legal tone)
  - Calibrate judge thresholds so it rewards fallback compliance, not just “relevance”
- **Potential behavior changes:**
  - Tool retry/backoff may increase request latency; ensure caps are enforced.
  - Auto-deriving missing params could be wrong if context interpretation is ambiguous—guard with “ask user” gates.
- **Genie fallback template safety:**
  - Alternate query templates must still be consistent with the user’s intent and constraints.
